# Grad-CAM for MNIST

Grad-CAM highlights image regions that most influenced a class score. This notebook implements a small version directly in PyTorch.

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

torch.manual_seed(42)

def get_device():
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"

device = get_device()

In [ ]:
class MNISTCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.relu = nn.ReLU()
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(16 * 7 * 7, 64), nn.ReLU(), nn.Linear(64, 10))

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        return self.classifier(x)

model = MNISTCNN().to(device)

In [ ]:
transform = transforms.ToTensor()
train_data = datasets.MNIST(root="../../data", train=True, download=True, transform=transform)
test_data = datasets.MNIST(root="../../data", train=False, download=True, transform=transform)
train_loader = DataLoader(Subset(train_data, range(5000)), batch_size=64, shuffle=True)
test_loader = DataLoader(Subset(test_data, range(1000)), batch_size=1)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(2):
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        loss = loss_fn(model(images), labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

In [ ]:
def grad_cam(model, image, target_class=None):
    activations = None
    gradients = None

    def forward_hook(module, inputs, output):
        nonlocal activations
        activations = output

    def backward_hook(module, grad_input, grad_output):
        nonlocal gradients
        gradients = grad_output[0]

    forward_handle = model.conv2.register_forward_hook(forward_hook)
    backward_handle = model.conv2.register_full_backward_hook(backward_hook)

    model.eval()
    model.zero_grad()
    logits = model(image)
    if target_class is None:
        target_class = logits.argmax(dim=1).item()
    logits[:, target_class].backward()

    weights = gradients.mean(dim=(2, 3), keepdim=True)
    cam = (weights * activations).sum(dim=1, keepdim=True)
    cam = F.relu(cam)
    cam = F.interpolate(cam, size=(28, 28), mode="bilinear", align_corners=False)
    cam = cam.squeeze().detach().cpu()
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

    forward_handle.remove()
    backward_handle.remove()
    return cam, target_class, F.softmax(logits.detach(), dim=1).squeeze().cpu()

In [ ]:
image, label = next(iter(test_loader))
image = image.to(device)
heatmap, predicted_class, probabilities = grad_cam(model, image)

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
axes[0].imshow(image.cpu().squeeze(), cmap="gray")
axes[0].set_title(f"input: {label.item()}")
axes[1].imshow(heatmap, cmap="magma")
axes[1].set_title(f"Grad-CAM for {predicted_class}")
axes[2].imshow(image.cpu().squeeze(), cmap="gray")
axes[2].imshow(heatmap, cmap="magma", alpha=0.45)
axes[2].set_title(f"confidence={probabilities[predicted_class]:.2f}")
for ax in axes:
    ax.axis("off")
plt.tight_layout()

## Exercise

Pass a different `target_class` into `grad_cam`. How does the highlighted region change when you ask for a class the model did not predict?